In [91]:
# imports
import numpy as np
import time
import pickle
import os
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [4]:
'''
The construction graph stores weight values, can evaporate them and normalize them.
'''
class ConstructionGraph:
    def __init__(self, num_of_items, num_of_bins, evaporation_rate):
        self.num_of_items = num_of_items
        self.num_of_bins = num_of_bins
        self.evaporation_rate = evaporation_rate
        
        np.random.seed(int(time.time()))
        self.start_weights = np.random.rand(self.num_of_bins)
        self.middle_weights = np.random.rand(self.num_of_items-1, self.num_of_bins, self.num_of_bins)
        
    '''
    A function for normalizing the weights such that the sum to 1, necessary for the selection of the next item
    '''
    def normalize(self):
        
         # normalize the weights
        self.start_weights = self.start_weights / np.sum(self.start_weights)
        self.middle_weights = self.middle_weights / np.sum(self.middle_weights, axis=1, keepdims=True)
        
        
    '''
    Evaporate the weights by multiplying with the evaporation rate e
    '''
    def evaporate(self):
        
        # reduce weights
        self.start_weights *= self.evaporation_rate
        self.middle_weights *= self.evaporation_rate
    
    '''
    debug code to print the normalized weights
    '''
    def show_weights(self):
        normalized_start_weights = self.start_weights / np.sum(self.start_weights)
        normalized_middle_weights = self.middle_weights / np.sum(self.middle_weights, axis=1, keepdims=True)
        print("Start weights: ", normalized_start_weights.tolist())
        print("Middle weights: ", normalized_middle_weights.tolist())

In [66]:

'''
The ant class, multiple instances are used and the same construction graph shared among ants

What it does
- initialises
- passes through the construction graph
- calculate the cost of each pass
'''
class Ant:
    def __init__(self, construction_graph, id, items):
        # legacy, used to help with seeding
        self.id = id
        
        # initialize items
        self.construction_graph = construction_graph
        self.items = items
        
        self.chosen = []
        self.cost = -1
        
    '''
    Let the ant go through the construction graph, and calculate the cost of the solution right after
    '''
    def forwardPass(self):
        
        # reset the chosen matrix
        self.chosen = []
        
        # seed and choose the first bin
        seed = time.time_ns() % 2**32
        np.random.seed(seed)
        
        # choose the first bin
        normalized_start_weights = self.construction_graph.start_weights / np.sum(self.construction_graph.start_weights)
        chosen_index = np.random.choice(self.construction_graph.num_of_bins, p=normalized_start_weights)
        self.chosen.append(chosen_index)
        
        # choose the next bins
        normalized_middle_weights = self.construction_graph.middle_weights / np.sum(self.construction_graph.middle_weights, axis=2, keepdims=True)
        for weight_matrix in normalized_middle_weights:
            chosen_index = np.random.choice(self.construction_graph.num_of_bins, p=weight_matrix[self.chosen[-1]])
            self.chosen.append(chosen_index)

		# calculate the cost after each forward pass
        self.calculate_cost(self.items)
        
    '''
    Calculate the cost of the current solution
    '''    
    def calculate_cost(self, items):
        
        # initialize the totals
        total = [0 for _ in range(self.construction_graph.num_of_bins)]
        
        # calculate the total for each bin
        for i in range(len(items)):
            total[self.chosen[i]] += items[i]
            
        # calculate the fitness
        self.cost = max(total) - min(total)
        
        return self.cost
    
    

In [75]:

'''
Handle all things ant colony related:

- initialise parameters
- update function
- traverse function
- save function
- evaluate function
- start function
'''
class ACO:
    def __init__(self, items, evaporation_rate, population_size, num_of_items, num_of_bins, save_dir, current_iteration, runWithoutMin):
        self.items = items
        self.population = []
        
        self.construction_graph = None
        
        self.population_size = population_size
        self.evaporation_rate = evaporation_rate
        
        self.num_of_items = num_of_items
        self.num_of_bins = num_of_bins
        
        self.save_dir = save_dir
        self.current_iteration = current_iteration
        self.evaluations = 0
        self.runWithoutMin = runWithoutMin
    
    '''
    Traverse all ants in population
    '''
    def traverseAll(self):

        # for each ant forward pass the graph
        for ant in self.population:
            ant.forwardPass()
            
    '''
    Legacy code
    '''    
    def parallelTraverseAll(self, ant):
        ant.forwardPass()
        return ant

    '''
    Update the pheromones in the construction graph based on the rule 100/fitness
    
    In cases where the cost is 0, which wont happen unless problem is very easy, the fitness is set to 1 to prevent division by 0
    '''
    def update(self):
        
        # the costs of the ants
        costs = [ant.cost for ant in self.population]
        
        # for every ant update reinforce pheromone
        for ant in self.population:
            
            # only update if the ant has the best cost
            if self.runWithoutMin or ant.cost == min(costs):
                
                # calculate fitness according to function, fitnesses of 0 break code, so 1
                fitness = ant.cost if ant.cost != 0 else 1
                fitness = 100 / fitness
                
                # update the start weights
                self.construction_graph.start_weights[ant.chosen[0]] += fitness
                
                # update the pheromones in the matrix
                for i in range(len(self.items)-1):
                    self.construction_graph.middle_weights[i][ant.chosen[i]][ant.chosen[i+1]] += fitness
                
            # increment the evaluations
            self.evaluations += 1
    '''
    Show the current state of the ACO and what the previous iteration of ants has chosen
    '''
    def evaluateAll(self):
        
        # initialise the values
        values = []
        
        for ant in self.population:
            
            # initialize the totals
            total = [0 for _ in range(self.construction_graph.num_of_bins)]
            
            # calculate the total for each bin
            for i in range(len(self.items)):
                total[ant.chosen[i]] += self.items[i]
                
            # update the values
            values.append(max(total) - min(total))
            
            print(total, end="")
            print(" - ", end="")
            
        
        print(sum(values) / len(values))
        
    '''
    Save the current progress of the ACO
    '''
    def save(self):
        log = []
        
        # check if the file has content already
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)
        
        # if the path exists then load previous content
        if os.path.exists(self.save_dir + str(self.current_iteration)):
            with open(self.save_dir + str(self.current_iteration), "rb") as f:
                log = pickle.load(f)
            
        # now add the current state
        log.append([ant.cost for ant in self.population])
        
        with open(self.save_dir + str(self.current_iteration), "wb+") as f:
            pickle.dump(log, f)
        
    '''
    The start function for the ACO, run after all parameters
    '''
    def start(self):
        
        # initialize the construction graph
        self.construction_graph = ConstructionGraph(self.num_of_items, self.num_of_bins, self.evaporation_rate)
        
        # initialize the population giving the same construction graph to all ants
        for i in range(self.population_size):
            ant = Ant(self.construction_graph, id=i+1, items=self.items)
            self.population.append(ant)
            
        # keep track of when to progress report
        iteration = 1
        
        # terminate on 10,000 fitness evaluations, 100 ants will run this 100 times, 10 ants will run this 1000 times
        while self.evaluations < 10000:
            
            # whoo let the ants out! WHO! Who! Who! Who!
            self.traverseAll()
            
            # update the pheromones
            self.update()
            
            # evaporate the construction graph a little
            self.construction_graph.evaporate() 

            # progress report
            if iteration % 10 == 0:
                print(f"iteration {iteration}", end=" - ")
                self.evaluateAll()
                
            # increase iterations
            iteration += 1
            
            # save progress at every step
            self.save()
            
        print("terminated")
            
            
        

In [ ]:

# the problem, commented out trials that run with max-min ( worse functionality )
parameters = [
#   (iter, item number, ants population, evaporation_rate, bins number, items, save_path)
	(5, 500, 100, 0.9, 10, [i+1 for i in range(500)], "progress/BPP1-5-100-0.9-10/", True),
	(5, 500, 100, 0.6, 10, [i+1 for i in range(500)], "progress/BPP1-5-100-0.6-10/", True),
	(5, 500, 10, 0.9, 10, [i+1 for i in range(500)], "progress/BPP1-5-10-0.9-10/", True),
	(5, 500, 10, 0.6, 10, [i+1 for i in range(500)], "progress/BPP1-5-10-0.6-10/", True),
	(5, 500, 100, 0.9, 50, [((i+1)**2)/2 for i in range(500)], "progress/BPP2-5-100-0.9-50/", True),
	(5, 500, 100, 0.6, 50, [((i+1)**2)/2 for i in range(500)], "progress/BPP2-5-100-0.6-50/", True),
	(5, 500, 10, 0.9, 50, [((i+1)**2)/2 for i in range(500)], "progress/BPP2-5-10-0.9-50/", True),
	(5, 500, 10, 0.6, 50, [((i+1)**2)/2 for i in range(500)], "progress/BPP2-5-10-0.6-50/", True),
 	# (5, 500, 100, 0.9, 10, [i+1 for i in range(500)], "progress-min/BPP1-5-100-0.9-10/", False),
	# (5, 500, 100, 0.6, 10, [i+1 for i in range(500)], "progress-min/BPP1-5-100-0.6-10/", False),
	# (5, 500, 10, 0.9, 10, [i+1 for i in range(500)], "progress-min/BPP1-5-10-0.9-10/", False),
	# (5, 500, 10, 0.6, 10, [i+1 for i in range(500)], "progress-min/BPP1-5-10-0.6-10/", False),
	# (5, 500, 100, 0.9, 50, [((i+1)**2)/2 for i in range(500)], "progress-min/BPP2-5-100-0.9-50/", False),
	# (5, 500, 100, 0.6, 50, [((i+1)**2)/2 for i in range(500)], "progress-min/BPP2-5-100-0.6-50/", False),
	# (5, 500, 10, 0.9, 50, [((i+1)**2)/2 for i in range(500)], "progress-min/BPP2-5-10-0.9-50/", False),
	# (5, 500, 10, 0.6, 50, [((i+1)**2)/2 for i in range(500)], "progress-min/BPP2-5-10-0.6-50/", False)
]

# some test parameters
# parameters2 = [
# 	(5, 50, 10, 0.9, 10, [i+1 for i in range(50)], "./progress/BPP1-5-10-0.9-10/"),
# 	(5, 50, 10, 0.6, 10, [i+1 for i in range(50)], "./progress/BPP1-5-10-0.6-10/"),
#  	(5, 50, 10, 0.9, 20, [((i+1)**2)/2 for i in range(50)], "./progress/BPP2-5-10-0.9-10/"),
# 	(5, 50, 10, 0.6, 20, [((i+1)**2)/2 for i in range(50)], "./progress/BPP2-5-10-0.6-10/"),
# ]

# a simple 2 bin 3 item problem with 2 possible weight configurations
# parameters3 = [
# 	(1, 3, 50, 0.9, 2, [i+1 for i in range(3)], "./progress/BPP1-1-10-0.9-2/", True),
#  	(1, 3, 50, 0.8, 2, [i+1 for i in range(3)], "./progress/BPP1-1-10-0.8-2/", True),
#   (1, 3, 50, 0.7, 2, [i+1 for i in range(3)], "./progress/BPP1-1-10-0.7-2/", True),
#   (1, 3, 50, 0.6, 2, [i+1 for i in range(3)], "./progress/BPP1-1-10-0.6-2/", True),
#   (1, 3, 50, 0.9, 2, [i+1 for i in range(3)], "./progress-min/BPP1-1-10-0.9-2/", False),
#  	(1, 3, 50, 0.8, 2, [i+1 for i in range(3)], "./progress-min/BPP1-1-10-0.8-2/", False),
#   (1, 3, 50, 0.7, 2, [i+1 for i in range(3)], "./progress-min/BPP1-1-10-0.7-2/", False),
#   (1, 3, 50, 0.6, 2, [i+1 for i in range(3)], "./progress-min/BPP1-1-10-0.6-2/", False),
# ]


# start the trials
for trial in parameters:
    
    # unpack the trial
    iterations = trial[0]
    num_of_items = trial[1]
    population_size = trial[2]
    evaporation_rate = trial[3]
    num_of_bins = trial[4]
    items = trial[5]
    save_dir = trial[6]
    run_without_min = trial[7]
    
    # run the trial for i iterations
    for i in range(iterations):
        aco = ACO(items=items, evaporation_rate=evaporation_rate, population_size=population_size, num_of_items=num_of_items, num_of_bins=num_of_bins, save_dir=save_dir, current_iteration=i, runWithoutMin=True)  
        aco.start()
        aco.evaluateAll()

In [ ]:

'''
This cell reads in all the ant data post execution and generates graphs for each parameter set

These graphs consist of 
- Each trial for a specific parameter set
'''


folders = []

# list of all folders in the progress directory
for folder in os.listdir("./progress"):
    folders.append(folder)
 
for folder_index in range(len(folders)):

	# start a new figure
    plt.figure()
    
    folder = folders[folder_index]
    files = []
    contents = []
    
    # list of all files in the folder
    for file in os.listdir(f"./progress/{folder}"):
        files.append(file)
    
    # read in all the data
    for i in range(5):
        contents.append(pickle.loads(open(f"./progress/{folder}/{i}", "rb").read()))

    # show each weight data for each iteration in a graph
    for iteration in contents:
        
        data = []
        
        # append data to list
        for graph in iteration:
            data.append(np.average(graph))
    
        # add line to graph
        plt.plot([i for i in range(len(data))], data, label=f"Iteration {i}")
        plt.xlabel("Iterations")
        plt.ylabel("Fitness")
        
    plt.title(f"Fitnesses {folder}")
    

# show everything
plt.show()

In [ ]:
'''
Purpose of this section of code is to show the best raw values of the fitnesses for each instance of the BPP

Values recorded are
- Best fitness value during the entire run for each trial
- Best Last fitness value for each trial
- Best Slope of the regression line for each trial
'''

# list of all folders in the progress directory
folders = []
for folder in os.listdir("./progress"):
    folders.append(folder)
 
# iterate over each folder and show the best values
for folder_index in range(len(folders)):
    
    folder = folders[folder_index]
    
    files = []
    trials = []
    
    # list of all files in the folder
    for file in os.listdir(f"./progress/{folder}"):
        files.append(file)
    
    # read in all the data
    for i in range(5):
        trials.append(pickle.loads(open(f"./progress/{folder}/{i}", "rb").read()))
    
    values = []
    last = []
    regression = []

    # show each weight data for each iteration in a graph
    for trial in trials:
        data = []
        
        # for every iteration in repeated calculation, will contain n amount of ants ( 10, 100 )
        for population in trial:     
            
            # get the best fitness
            data.append(min(population))
        
        # append the minimum and last value of the trial to the list
        values.append(min(data))
        last.append(data[-1])
        
        # remove converged ants
        while len(data) > 1 and data[-1] == data[-2]:
            data.pop()
        

        # Create time indices
        time_index = np.arange(len(data)).reshape(-1, 1)

        # Fit a linear regression model with removed duplicates
        model = LinearRegression().fit(time_index, data)

        # Check the slope of the regression line
        slope = model.coef_[0]

        regression.append(slope)  # True if the slope is negative (indicating a decreasing trend)
    
    # show best values for each instance of BPP
    print(f"\n\n{folder}")
    print(min(values))
    print(min(last))
    print(min(regression))